# Fixations Dataset

Dataset key: `model_test/real_data_sets/fixations_dataset`

This is the historical fixation-locked real dataset. It keeps its original HDF5 layout, so this notebook loads it through the plotting helper's fixation-specific functions.

In [ ]:
import Pkg

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_19", "data_sources")
const DATASETS_ROOT = joinpath(REPO_ROOT, "notebooks", "datasets")
const WEEK19_DOWNLOADS = joinpath(REPO_ROOT, "notebooks", "week_19", "downloads")
const PYTHON = begin
    venv_python = joinpath(REPO_ROOT, ".venv_8bit", "bin", "python")
    isfile(venv_python) ? venv_python : "python"
end

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using CSV
using DataFrames
using HDF5
using JSON3
using Printf
using Statistics

include(joinpath(REPO_ROOT, "notebooks", "week_15", "try_new_data_helpers.jl"))
using .Week15TryNewData

mkpath(WEEK19_DOWNLOADS)
RNG_SEED = Int(mod(time_ns(), UInt64(typemax(Int))))
println("Repo root: ", REPO_ROOT)
println("Python: ", PYTHON)
println("RNG seed: ", RNG_SEED)

In [ ]:
const FIXATION_DIR = joinpath(REPO_ROOT, "notebooks", "model_test", "real_data_sets", "fixations_dataset")
const FIXATION_H5 = joinpath(FIXATION_DIR, "data_fixations.hdf5")
const FIXATION_EVENTS = joinpath(FIXATION_DIR, "events.csv")

@assert isfile(FIXATION_H5) "Missing fixation HDF5: $FIXATION_H5"
@assert isfile(FIXATION_EVENTS) "Missing fixation events: $FIXATION_EVENTS"

println("Fixation HDF5: ", FIXATION_H5)
println("Events CSV: ", FIXATION_EVENTS)
display(fixation_summary_df())

sort_audit = fixation_sort_order_audit_df()
display(sort_audit)
@assert all(sort_audit.status .== "ok") "Sort-order audit failed for fixation reference data."


In [ ]:
const TARGET_SIZE = nothing
const N_SAMPLES_PER_SORT = 16
const N_COLS = 4
fixation_cache = load_fixation_reference_cache(
    per_sort_var = N_SAMPLES_PER_SORT,
    target_size = TARGET_SIZE,
    rng_seed = RNG_SEED,
    preferred_only = false,
)

fixation_plot_summary = combine(groupby(fixation_cache.meta, :sort_var), nrow => :n_images)
display(fixation_plot_summary)
@assert all(fixation_plot_summary.n_images .== N_SAMPLES_PER_SORT) "Expected $(N_SAMPLES_PER_SORT) plots per sort variable."

for sort_var in fixation_plot_summary.sort_var
    display(plot_fixation_reference_grid(fixation_cache; sort_var = sort_var, n_cols = N_COLS))
end
